# 检索增强生成 (RAG) 与混合检索重排序企业笔试手撕通关宝典
> **面向对象**：互联网大厂/AI 独角兽企业 检索增强生成 (RAG) / 搜索推荐 / 向量检索 / 知识库问答算法岗面试手撕  
> **核心涵盖**：滑动窗口文档切分、BM25 倒排与打分公式、稠密向量检索、倒数排名融合 (RRF)、Cross-Encoder 交叉注意力精排、HyDE 假设性文档嵌入、HitRate/MRR/NDCG 评测指标、多轮对话 Prompt 动态组装  
> **设计准则**：纯 Python / NumPy / PyTorch 极简手撕，彻底扫除第三方向量库黑盒，吃透多路召回与重排精髓。

---
### 核心模块速览
1. **模块一**：滑动窗口带重叠文档切分器 (Chunking with Overlap) 纯 Python 手撕
2. **模块二**：BM25 词频检索器纯 NumPy 手撕 (倒排索引、TF 饱和度与长度惩罚)
3. **模块三**：稠密向量检索 (Dense Retrieval) 与高维余弦相似度 Top-K 手撕
4. **模块四**：倒数排名融合算法 (Reciprocal Rank Fusion, RRF) 手撕
5. **模块五**：Cross-Encoder 交叉编码器精排单挑打分模拟手撕
6. **模块六**：假设性文档嵌入 (HyDE, Hypothetical Document Embeddings) 机制手撕
7. **模块七**：RAG 工业评估指标全矩阵手撕 (HitRate@K, MRR@K, NDCG@K)
8. **模块八**：多轮对话指代消解与 Prompt 动态组装器手撕

---
## 模块一：滑动窗口带重叠文档切分器 (Chunking with Overlap) 纯 Python 手撕

### 【笔试考点与物理边界】
1. **切分必然性**：Cross-Encoder 全注意力机制 $O(N^2)$ 算力限制，无法直接吞咽整本长文档；
2. **滑动窗口步长计算**：
   $$\text{stride} = \text{chunk\_size} - \text{chunk\_overlap}$$
   保留重叠防止核心实体、主谓宾在切断处被硬生生撕裂导致召回率死穴。

In [ ]:
def sliding_window_chunker(text, chunk_size=50, chunk_overlap=10):
    """
    滑动窗口切块器
    text: 输入长文本
    chunk_size: 切片窗口字符长度
    chunk_overlap: 重叠保留长度
    """
    assert chunk_size > chunk_overlap, "chunk_size 必须大于 chunk_overlap!"
    stride = chunk_size - chunk_overlap
    chunks = []
    
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        if end >= len(text):
            break
        start += stride
    return chunks

# 测试文档切分
sample_text = "检索增强生成(RAG)技术通过结合外部检索库与大语言模型生成能力，能够显著降低幻觉并增强时效性知识储备。"
chunks = sliding_window_chunker(sample_text, chunk_size=20, chunk_overlap=5)
print(f"原始文本长度 {len(sample_text)}, 切出 {len(chunks)} 个切片:")
for i, c in enumerate(chunks):
    print(f"  Chunk {i}: {c}")
assert len(chunks) > 1
print(">>> 滑动窗口重叠切分器验证通过！")

---
## 模块二：BM25 词频检索器纯 NumPy 手撕 (倒排索引与长度惩罚)

### 【笔试顶级高频考点】
1. **逆文档频率 (IDF)**：
   $$\text{IDF}(q_i) = \ln \left( \frac{N - n(q_i) + 0.5}{n(q_i) + 0.5} + 1 \right)$$
2. **词频饱和度与文档长度归一化**：
   $$\text{Score}(D, Q) = \sum_{q_i \in Q} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left( 1 - b + b \cdot \frac{|D|}{\text{avgdl}} \right)}$$
   - $k_1$：词频饱和度参数（通常 1.2~2.0）；
   - $b$：文档长度惩罚系数（通常 0.75）。长文档因字数多天然词频高，被分母狠狠压制。

In [ ]:
import math
import collections
import numpy as np

class SimpleBM25:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.corpus = corpus
        self.N = len(corpus)
        self.doc_lens = [len(doc) for doc in corpus]
        self.avgdl = sum(self.doc_lens) / max(1, self.N)
        
        # 1. 建立倒排索引与词频统计
        self.doc_freqs = collections.defaultdict(int) # df
        self.doc_term_counts = []                     # tf per doc
        
        for doc in corpus:
            counts = collections.Counter(doc)
            self.doc_term_counts.append(counts)
            for word in counts.keys():
                self.doc_freqs[word] += 1
                
        # 2. 预计算每个词的 IDF
        self.idf = {}
        for word, df in self.doc_freqs.items():
            self.idf[word] = math.log((self.N - df + 0.5) / (df + 0.5) + 1.0)

    def score(self, query):
        """计算 Query 针对全库各文档的 BM25 评分向量"""
        scores = np.zeros(self.N)
        for q in query:
            if q not in self.idf:
                continue
            idf_val = self.idf[q]
            for doc_idx in range(self.N):
                tf = self.doc_term_counts[doc_idx].get(q, 0)
                len_norm = 1.0 - self.b + self.b * (self.doc_lens[doc_idx] / self.avgdl)
                tf_term = (tf * (self.k1 + 1.0)) / (tf + self.k1 * len_norm)
                scores[doc_idx] += idf_val * tf_term
        return scores

# 测试 BM25
toy_corpus = [
    ["苹果", "公司", "发布", "新", "手机"],
    ["华为", "手机", "通信", "技术", "领先"],
    ["苹果", "营养", "丰富", "多", "吃", "水果"]
]
bm25 = SimpleBM25(toy_corpus)
scores = bm25.score(["苹果", "手机"])
print("BM25 对三篇文档的打分:", np.round(scores, 4))
best_doc_idx = np.argmax(scores)
print(f"Top-1 命中契合度最高的文档 ID: {best_doc_idx} 内容: {toy_corpus[best_doc_idx]}")
assert best_doc_idx == 0
print(">>> BM25 稀疏精确检索器验证成功！")

---
## 模块三：稠密向量检索 (Dense Retrieval) 与高维余弦相似度 Top-K 手撕

### 【笔试考点】
- **余弦相似度**：
  $$\cos(u, v) = \frac{u \cdot v}{\|u\|_2 \|v\|_2}$$
- **全矩阵向量化**：先对 Query 和所有 Document 向量进行 L2 归一化，则余弦相似度直接简化为矩阵乘法：$\text{Sims} = Q_{\text{norm}} \cdot D_{\text{norm}}^T$！

In [ ]:
def dense_retrieval_topk(query_vec, doc_vecs, top_k=2):
    """
    query_vec: (D,)
    doc_vecs: (N, D)
    """
    # L2 归一化
    q_norm = query_vec / (np.linalg.norm(query_vec) + 1e-8)
    d_norm = doc_vecs / (np.linalg.norm(doc_vecs, axis=-1, keepdims=True) + 1e-8)
    
    # 矩阵乘法点乘即为余弦相似度
    sims = np.dot(d_norm, q_norm) # (N,)
    topk_indices = np.argsort(sims)[::-1][:top_k]
    topk_scores = sims[topk_indices]
    return topk_indices, topk_scores

# 测试稠密检索
doc_embeddings = np.array([
    [0.9, 0.1, 0.0], # 文档 0: 科技类
    [0.0, 0.8, 0.2], # 文档 1: 水果类
    [0.85, 0.2, 0.1] # 文档 2: 科技类
])
query_embedding = np.array([1.0, 0.0, 0.0]) # 科技 query

topk_ids, topk_scs = dense_retrieval_topk(query_embedding, doc_embeddings, top_k=2)
print("Dense 检索 Top-K 索引:", topk_ids)
print("Dense 检索对应相似度:", np.round(topk_scs, 4))
assert topk_ids[0] == 0 and topk_ids[1] == 2
print(">>> 稠密向量全矩阵检索验证成功！")

---
## 模块四：倒数排名融合算法 (Reciprocal Rank Fusion, RRF) 纯手撕

### 【笔试高频考点】
1. **量纲鸿沟**：BM25 打分可能为 30 分，Dense 余弦相似度卡在 `[-1, 1]`，**绝对严禁直接加权求和**；
2. **RRF 闭式解**：抛弃绝对分数，只看名次 $r$：
   $$\text{RRF\_Score}(d) = \sum_{m \in \text{methods}} \frac{1}{k + r_m(d)}$$
   常数 $k$ 工业界惯用 **$60$**，有效防止头牌极低排名带来的过度支配。

In [ ]:
def reciprocal_rank_fusion(ranked_lists, k=60):
    """
    倒数排名融合 RRF
    ranked_lists: 包含多个检索器排序结果的列表，如 [[docA, docB], [docB, docC]]
    """
    rrf_scores = collections.defaultdict(float)
    
    for rank_list in ranked_lists:
        for rank, doc_id in enumerate(rank_list, start=1):
            rrf_scores[doc_id] += 1.0 / (k + rank)
            
    # 按综合得分降序排序
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_docs

# 测试双路召回融合
bm25_ranking = ["Doc_A", "Doc_B", "Doc_C"] # BM25 排行
dense_ranking = ["Doc_B", "Doc_A", "Doc_D"] # 稠密向量排行

merged_ranks = reciprocal_rank_fusion([bm25_ranking, dense_ranking], k=60)
print("RRF 多路召回融合后榜单得分:")
for doc, score in merged_ranks:
    print(f"  {doc}: {score:.5f}")
# Doc_A 和 Doc_B 都在前列，且均优于只有单路入选的 Doc_C/Doc_D
assert merged_ranks[0][0] in ["Doc_A", "Doc_B"]
print(">>> 倒数排名融合 (RRF) 算法验证通过！")

---
## 模块五：Cross-Encoder 交叉编码器精排单挑打分模拟手撕

### 【笔试必问核心原理】
- **Bi-Encoder (双塔)**：Query 与 Doc 分开编码，适合毫秒级初排百万召回；
- **Cross-Encoder (单塔交叉)**：`[CLS] Query [SEP] Chunk [SEP]` 全程做多层 Self-Attention 全交互，精度封顶但算力昂贵，仅用于精排几十个候选切片！

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MockCrossEncoderReranker(nn.Module):
    def __init__(self, d_model=32):
        super().__init__()
        # 模拟 Transformer 顶层打分头: 从 [CLS] 输出映射为单个对齐标量
        self.score_head = nn.Linear(d_model, 1)

    def forward(self, cls_representations):
        """
        cls_representations: (B, d_model) Query 与各 Chunk 拼接后提取的 [CLS] 向量
        """
        raw_scores = self.score_head(cls_representations).squeeze(-1) # (B,)
        probs = torch.sigmoid(raw_scores)                             # 映射为 [0, 1] 关联度
        return probs

reranker = MockCrossEncoderReranker(d_model=16)
# 模拟 3 个候选 chunk 的 CLS 交互表征
mock_cls = torch.randn(3, 16)
re_scores = reranker(mock_cls)
print("Cross-Encoder 二次精排关联度打分 (范围 [0, 1]):", re_scores.detach().numpy())
assert re_scores.shape == (3,) and (re_scores <= 1.0).all()
print(">>> Cross-Encoder 交叉重排序单挑模拟通过！")

---
## 模块六：假设性文档嵌入 (HyDE, Hypothetical Document Embeddings) 机制手撕

### 【笔试考点与流程】
- **原理**：短 Query（几个词）与候选长文档（几百词）存在严重的长度与语义流形不对称；
- **HyDE 流程**：
  $$\text{Query} \xrightarrow{\text{Zero-Shot LLM}} \text{假答案 (Hypothetical Doc)} \xrightarrow{\text{Embedding}} \text{向量} \xrightarrow{\text{搜索}} \text{真实文档向量}$$
  拿着“答案的向量”去库里搜“真答案”，跨越提问与陈述的语义鸿沟！

In [ ]:
def hyde_pipeline_mock(query, mock_llm_fn, embedding_fn, vector_store_fn):
    """HyDE 核心流程管道"""
    # 1. 大模型先脑补一段假答案
    hypothetical_doc = mock_llm_fn(query)
    print("HyDE 脑补的假答案:", hypothetical_doc)
    
    # 2. 对假答案进行向量化
    hypo_vec = embedding_fn(hypothetical_doc)
    
    # 3. 拿着假答案向量检索真文档
    retrieved_docs = vector_store_fn(hypo_vec)
    return retrieved_docs

mock_llm = lambda q: f"{q} 的核心原理是基于外部数据库知识检索并注入大模型上下文生成回答。"
mock_embed = lambda text: np.random.randn(8)
mock_store = lambda vec: ["Doc_1: RAG 工业架构全景", "Doc_2: 知识库问答实战"]

res_docs = hyde_pipeline_mock("什么是 RAG？", mock_llm, mock_embed, mock_store)
print("HyDE 检索出的最终真实文档:", res_docs)
assert len(res_docs) == 2
print(">>> 假设性文档嵌入 (HyDE) 验证成功！")

---
## 模块七：RAG 工业评估指标全矩阵手撕 (HitRate@K, MRR@K, NDCG@K)

### 【笔试顶级评估硬核指标】
1. **HitRate@K**：黄金文档是否进入前 $K$（0 或 1）；
2. **MRR@K (平均倒数排名)**：
   $$\text{MRR@K} = \frac{1}{Q} \sum_{i=1}^Q \frac{1}{\text{rank}_i}$$
3. **NDCG@K (归一化折损累计增益)**：
   $$\text{DCG@K} = \sum_{i=1}^K \frac{2^{\text{rel}_i} - 1}{\log_2(i + 1)}, \quad \text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}$$

In [ ]:
def evaluate_rag_metrics(retrieved_ids, ground_truth_id, k=3):
    """
    计算单样本的 HitRate@K, MRR@K, NDCG@K
    retrieved_ids: 检索返回的文档 ID 列表
    ground_truth_id: 黄金标准相关文档 ID
    """
    topk_list = retrieved_ids[:k]
    
    # 1. HitRate@K
    hit = 1.0 if ground_truth_id in topk_list else 0.0
    
    # 2. MRR@K
    if ground_truth_id in topk_list:
        rank = topk_list.index(ground_truth_id) + 1
        mrr = 1.0 / rank
    else:
        mrr = 0.0
        
    # 3. NDCG@K (二值相关性: 命中文档增益为 1，其余为 0)
    dcg = 0.0
    for i, doc_id in enumerate(topk_list, start=1):
        rel = 1.0 if doc_id == ground_truth_id else 0.0
        dcg += (2.0 ** rel - 1.0) / math.log2(i + 1)
        
    # 理想 DCG (最佳情况下第 1 位命中)
    idcg = (2.0 ** 1.0 - 1.0) / math.log2(1 + 1)
    ndcg = dcg / idcg
    
    return hit, mrr, ndcg

# 测试评测指标: 假设真实文档排在第 2 位
cand_ids = ["doc_99", "doc_gold", "doc_45"]
hit, mrr, ndcg = evaluate_rag_metrics(cand_ids, ground_truth_id="doc_gold", k=3)
print(f"HitRate@3: {hit:.1f} (命中)")
print(f"MRR@3: {mrr:.4f} (排在第 2 位即 1/2)")
print(f"NDCG@3: {ndcg:.4f}")
assert hit == 1.0 and np.isclose(mrr, 0.5)
print(">>> RAG 工业级评估指标手撕验证成功！")

---
## 模块八：多轮对话指代消解与 Prompt 动态组装器手撕

### 【笔试考点与状态机】
- **CondensePlusContextMode 机制**：将历史聊天记录与当轮指代不明的问题（如“它的价格是多少？”）组合，一方面重写为独立搜索查询（“苹果 iPhone 15 的价格是多少？”），另一方面将搜索到的上下文片段无缝注入 Prompt 供模型回答。

In [ ]:
class RAGPromptBuilder:
    def __init__(self, system_instruction):
        self.system_instruction = system_instruction

    def condense_question(self, chat_history, raw_question):
        """简单模拟多轮对话指代消解重写"""
        if not chat_history:
            return raw_question
        last_entity = "iPhone 16" # 假定上文提到了 iPhone 16
        if "它" in raw_question:
            return raw_question.replace("它", last_entity)
        return raw_question

    def build_augmented_prompt(self, user_question, retrieved_contexts):
        context_str = "\n".join([f"[{i+1}] {c}" for i, c in enumerate(retrieved_contexts)])
        prompt = (
            f"系统设定: {self.system_instruction}\n\n"
            f"参考资料库 (证据文档):\n{context_str}\n\n"
            f"用户问题: {user_question}\n\n"
            f"请严格根据上述资料库回答用户问题，严禁产生幻觉："
        )
        return prompt

builder = RAGPromptBuilder("你是一个严谨的企业级专业智能客服。")
rewritten = builder.condense_question(["Q: 聊聊 iPhone 16", "A: 好的"], "它的官方售价是多少？")
print("多轮指代消解重写后独立 Query:", rewritten)
assert "iPhone 16" in rewritten

final_p = builder.build_augmented_prompt(rewritten, ["iPhone 16 官方起售价为 5999 元。"])
print("\n动态生成的完整增强 Prompt:\n", final_p)
assert "5999 元" in final_p
print(">>> 多轮对话 Prompt 组装器验证成功！")

---
## 企业笔试手撕核心口诀与雷区速记卡

```
1. 滑动重叠切分: stride = chunk_size - overlap，重叠保留防主谓宾断裂召回流失。
2. BM25 长度惩罚: 词频加 k1 饱和平滑，分母除以 (1 - b + b * len/avgdl) 压制水货长文。
3. 倒数排名融合: rrf = sum(1 / (60 + rank))，抛弃绝对分数差异，多路召回无偏对齐。
4. 交叉单挑精排: [CLS] Query [SEP] Doc [SEP] 做深层注意力，算力贵但精度极致。
5. 评测三大金刚: HitRate 看是否中，MRR 考倒数名次，NDCG 折损累计对数增益。
```